# Time Series Forecasting with AutoGluon

AutoGluon's `TimeSeriesPredictor` fits a suite of forecasting models (statistical, tree-based, and deep learning) and returns **probabilistic** forecasts — a median plus confidence quantiles.

This notebook covers:
1. The `TimeSeriesDataFrame` — AutoGluon's required data format
2. Fitting a predictor and reading probabilistic forecasts
3. Evaluating forecast quality
4. Working with multiple time series
5. Practical gotchas

In [ ]:
import autogluon
print('AutoGluon version:', autogluon.__version__)

## 1. The TimeSeriesDataFrame

AutoGluon requires data in its own `TimeSeriesDataFrame` format — a pandas DataFrame with a **MultiIndex** `(item_id, timestamp)`. Every row is one observation for one series at one timestamp.

Required columns in your source DataFrame:
- An **item_id** column: identifies which time series each row belongs to
- A **timestamp** column: must be parseable as `datetime64`
- A **target** column: the value you want to forecast

In [ ]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# Load the M4 hourly sample dataset provided by AutoGluon
df_raw = pd.read_csv(
    'https://autogluon.s3.amazonaws.com/datasets/timeseries/m4_hourly_subset/train.csv'
)
print(df_raw.shape)
df_raw.head()

In [ ]:
# The CSV already has item_id and timestamp columns — convert to TimeSeriesDataFrame
train_data = TimeSeriesDataFrame.from_data_frame(
    df_raw,
    id_column='item_id',
    timestamp_column='timestamp',
)

print('Type:', type(train_data))
print('Index levels:', train_data.index.names)
print('Number of series:', train_data.num_items)
train_data.head(8)

In [ ]:
# Visualise one series
import matplotlib.pyplot as plt

item = train_data.item_ids[0]
series = train_data.loc[item]
series['target'].plot(title=f'Series: {item}', figsize=(12, 3))
plt.tight_layout()
plt.show()

## 2. Fit the Predictor

Key parameters set **at init time** (not fit time):
- **`prediction_length`** — how many future steps to forecast (e.g., 48 = 48 hours ahead)
- **`eval_metric`** — `'MASE'` (scale-invariant), `'MAPE'`, `'WQL'` (weighted quantile loss)
- **`freq`** — data frequency; AutoGluon tries to infer it, but set explicitly if unsure

> **`prediction_length` must be set before `fit()`** — you cannot change it later without retraining.

In [ ]:
predictor = TimeSeriesPredictor(
    prediction_length=48,        # forecast 48 hours ahead
    eval_metric='MASE',
    path='./ag_timeseries_model',
    target='target',             # column with values to predict (default: 'target')
    freq='h',                    # hourly; set explicitly to avoid inference errors
)

predictor.fit(
    train_data=train_data,
    time_limit=120,
    presets='fast_training',     # 'fast_training', 'medium_quality', 'best_quality'
    verbosity=1,
)

## 3. Make Forecasts

`.predict()` returns a `TimeSeriesDataFrame` with **quantile columns** — a range of outcomes, not just a point estimate. The `'0.5'` column is the median (best single-value forecast).

In [ ]:
predictions = predictor.predict(train_data)

print('Forecast columns:', predictions.columns.tolist())
print('Number of series forecast:', predictions.num_items)
predictions.head()

In [ ]:
# Plot observed history + forecast + confidence band for one series
fig, ax = plt.subplots(figsize=(14, 4))

item = predictions.item_ids[0]
history = train_data.loc[item]['target'].iloc[-100:]  # last 100 observations
forecast = predictions.loc[item]

history.plot(ax=ax, label='Observed', color='black')
forecast['0.5'].plot(ax=ax, label='Median forecast', color='tab:blue')
ax.fill_between(
    forecast.index,
    forecast['0.1'],
    forecast['0.9'],
    alpha=0.2,
    color='tab:blue',
    label='10-90% interval',
)
ax.legend()
ax.set_title(f'Forecast for {item}')
plt.tight_layout()
plt.show()

## 4. Evaluate Forecast Quality

In [ ]:
# Load the held-out test data
df_test_raw = pd.read_csv(
    'https://autogluon.s3.amazonaws.com/datasets/timeseries/m4_hourly_subset/test.csv'
)
test_data = TimeSeriesDataFrame.from_data_frame(
    df_test_raw,
    id_column='item_id',
    timestamp_column='timestamp',
)

# evaluate() expects test_data to contain FULL history + the forecast window
scores = predictor.evaluate(test_data)
print('Evaluation metrics:', scores)

In [ ]:
# Model leaderboard
predictor.leaderboard().head(10)

## 5. Single-Series Example

Even when you have only one time series, you must still assign an `item_id`.

In [ ]:
import numpy as np

# Create a simple single-series dataset from scratch
dates = pd.date_range('2020-01-01', periods=365, freq='D')
values = 100 + np.cumsum(np.random.randn(365))  # random walk

single_df = pd.DataFrame({
    'item_id': 'my_series',   # even a single series needs an ID
    'timestamp': dates,
    'target': values,
})

single_ts = TimeSeriesDataFrame.from_data_frame(
    single_df,
    id_column='item_id',
    timestamp_column='timestamp',
)

single_predictor = TimeSeriesPredictor(
    prediction_length=30,
    freq='D',
    path='./ag_single_series',
)
single_predictor.fit(single_ts, time_limit=60, presets='fast_training', verbosity=0)

single_forecast = single_predictor.predict(single_ts)
print('Forecast shape:', single_forecast.shape)
single_forecast.head()

## ⚠️ Practical Gotchas

| Gotcha | What Happens | Fix |
|--------|-------------|-----|
| **`prediction_length` is set at init, not fit** | Changing it requires full retraining | Decide on your forecast horizon before creating the predictor |
| **Timestamps are strings, not datetime** | Frequency inference fails silently or errors | Run `df['timestamp'] = pd.to_datetime(df['timestamp'])` before converting |
| **Irregular timestamps / gaps** | Models assume regular intervals; gaps cause errors | Resample/interpolate to a regular grid first |
| **Series too short** | Need at least `2 × prediction_length` observations per series | Filter out short series before fitting |
| **`item_id` missing from test data** | AutoGluon cannot forecast series it did not see in training | All `item_id`s in test must appear in training |
| **Forecast output columns are strings `'0.1'`, `'0.5'`, `'0.9'`** | Trying `predictions[0.5]` raises `KeyError` | Use `predictions['0.5']` (string key) |
| **`evaluate()` needs the full test window** | Passing only the future horizon causes wrong scores | Pass the full history + future horizon together as `test_data` |
| **One series in multi-series data** | Tree models (LightGBM) do not learn cross-series patterns with <3 series | Add more series or use statistical models only |